# E7 — Leave-one-source-out: measuring the domain gap BETWEEN PPE-Unified's sources (YOLO26s, 15 epochs per run)

**Question:** E3 measured the domain gap OUTSIDE the benchmark (SH17: −70 to −88 points). How large is the gap INSIDE it
— between the three sources A (helmet), B (vest/boots) and C (gloves/goggles/mask)? Answering this quantifies
how much each class depends on its source's "image style", feeding directly into the domain-bias analysis.

**Design (3 rotating runs):**

| Run | Train + Val | Test (zero-shot) | Measurable classes |
|---|---|---|---|
| `loso_A` | B + C | all source-A images | helmet, no_helmet |
| `loso_B` | A + C | all source-B images | vest/no_vest, boots/no_boots (B-exclusive) + helmet |
| `loso_C` | A + B | all source-C images | the gloves/goggles/mask pairs (C's core classes) |

- Images are filtered by the `source` column of `manifest.csv`; train/val use the original split of the two retained sources (no re-splitting);
  test = ALL images of the held-out source (every split — valid because the model has never seen that source, so there is no leakage).
- AP is scored only for classes with ≥50 boxes in that run's test set (absent classes → NaN, excluded from the mean).
- Same YOLO26s, seed 42, lr0 0.01, batch 16, 15 epochs, RFS off.

**Output:** `results/e7_loso.csv` — per run: mAP@50 (over classes with enough boxes), per-class AP for the main classes, and the gap versus in-distribution.
Expected reading: classes present in only one source (vest, boots) should drop the most — evidence that "source diversity beats sample count".

**Add Input:** `ppe-unified-v1`. GPU T4 x2 · Internet ON. Estimated ~5-6 h (3 runs). Checkpoint-resume supported.

In [ ]:
KEY_PREFIX = "e7_"
# ==================== CONFIG ====================
QUICK_TEST = False             # True = 1 epoch / a few batches as a pipeline smoke test; quick numbers must NOT go into the report
SEED = 42

import os, random, shutil, glob, json, time
WORK = "/kaggle/working"; os.makedirs(f"{WORK}/results", exist_ok=True)
STATE_PATH = f"{WORK}/state.json"
# RESUME: load state.json from THIS notebook's previous output (if Added as Input)
STATE = {}
for _st in glob.glob("/kaggle/input/**/state.json", recursive=True):
    try: _s = json.load(open(_st))
    except Exception: continue
    if any(k.startswith(KEY_PREFIX) for k in _s): STATE.update(_s); print("Resumed state:", list(_s))
def save_state():
    json.dump(STATE, open(STATE_PATH, "w"), ensure_ascii=False, indent=1)
def done(key): return key in STATE
random.seed(SEED)


In [ ]:
%pip install -q ultralytics
import torch; assert torch.cuda.is_available(), "Enable GPU T4 x2!"
print("GPU:", torch.cuda.get_device_name(0), "- OK")


In [ ]:
from pathlib import Path
import yaml, pandas as pd

def find_dataset_root():
    """Locate PPE_Unified_v1 under /kaggle/input (recursive glob — no hard-coded path)."""
    for c in glob.glob("/kaggle/input/**/data.yaml", recursive=True):
        root = os.path.dirname(c)
        if all(os.path.isdir(f"{root}/{s}/images") for s in ("train", "val", "test")):
            return root
    raise AssertionError("Dataset ppe-unified-v1 has not been Added as Input!")

def find_manifest():
    """manifest.csv traces each image back to its origin (source A/B/C) — produced by build_unified_dataset.py."""
    for c in glob.glob("/kaggle/input/**/manifest.csv", recursive=True):
        return pd.read_csv(c)
    raise AssertionError("manifest.csv not found in the input dataset!")

DS_ROOT = find_dataset_root()
MANIFEST = find_manifest()
print("Dataset:", DS_ROOT, "| manifest:", len(MANIFEST), "rows")
print(MANIFEST["source"].value_counts() if "source" in MANIFEST.columns else MANIFEST.columns.tolist())


In [ ]:
# ---------- Build the LOSO datasets ----------
def build_loso(dst_name, held_out):
    """Train/val = the two remaining sources (original split kept); test = ALL images of the held-out source."""
    dst = f"{WORK}/{dst_name}"
    if os.path.isdir(dst): shutil.rmtree(dst)
    for sp in ("train", "val", "test"):
        for sub in ("images", "labels"):
            os.makedirs(f"{dst}/{sp}/{sub}", exist_ok=True)
    for _, row in MANIFEST.iterrows():
        src_letter = str(row["source"]).upper()[0]
        split, fname = row["split"], row["file"]
        lbl = fname.rsplit(".", 1)[0] + ".txt"
        # held-out source -> everything goes to test; retained sources -> original split minus their test split
        # (the retained sources' test split is unused — preserving the rule that TEST is touched once, in the main notebooks)
        if src_letter == held_out:
            tgt = "test"
        elif split in ("train", "val"):
            tgt = split
        else:
            continue
        shutil.copy(f"{DS_ROOT}/{split}/images/{fname}", f"{dst}/{tgt}/images/{fname}")
        shutil.copy(f"{DS_ROOT}/{split}/labels/{lbl}", f"{dst}/{tgt}/labels/{lbl}")
    names = yaml.safe_load(open(f"{DS_ROOT}/data.yaml"))["names"]
    dyaml = f"{dst}/data.yaml"
    yaml.dump({"path": dst, "train": "train/images", "val": "val/images", "test": "test/images",
               "nc": len(names), "names": names}, open(dyaml, "w"))
    return dyaml


In [ ]:
# ---------- 3 run LOSO ----------
from ultralytics import YOLO
from collections import Counter
import pandas as pd
EPOCHS = 1 if QUICK_TEST else 15
MIN_BOX = 50   # classes below this count in test -> NaN, excluded from the run's mean mAP
rows = []
for held in ["A", "B", "C"]:
    key = f"e7_loso_{held}"
    if done(key): rows.append(STATE[key]); print("skip:", key); continue
    dyaml = build_loso(f"ds_loso_{held}", held)
    # count test boxes per class to apply the threshold
    cnt = Counter()
    for lf in glob.glob(f"{WORK}/ds_loso_{held}/test/labels/*.txt"):
        for line in open(lf):
            if line.strip(): cnt[int(line.split()[0])] += 1
    t0 = time.time()
    m = YOLO("yolo26s.pt")
    m.train(data=dyaml, epochs=EPOCHS, imgsz=640, batch=16, seed=SEED, lr0=0.01,
            project=f"{WORK}/runs", name=key, exist_ok=True, plots=False, verbose=False)
    t = m.val(data=dyaml, split="test", batch=16)
    names = list(t.names.values())
    valid, per = [], {}
    for i, (n, a) in enumerate(zip(names, t.box.ap50)):
        if cnt.get(i, 0) >= MIN_BOX:
            per[f"AP50_{n}"] = round(float(a), 4); valid.append(float(a))
    rec = {"held_out": held, "n_test_img": len(glob.glob(f"{WORK}/ds_loso_{held}/test/images/*")),
           "mAP50_valid_classes": round(sum(valid) / max(len(valid), 1), 4),
           "n_valid_classes": len(valid), **per,
           "min": round((time.time() - t0) / 60, 1)}
    STATE[key] = rec; save_state(); rows.append(rec)
    shutil.rmtree(f"{WORK}/ds_loso_{held}", ignore_errors=True)
df = pd.DataFrame(rows); df.to_csv(f"{WORK}/results/e7_loso.csv", index=False)
print(df.to_string(index=False))


In [ ]:
# ==================== CLEAN UP OUTPUT: no dataset bundled ====================
KEEP = {"results", "state.json", "runs"}
for item in os.listdir(WORK):
    if item in KEEP: continue
    p = os.path.join(WORK, item)
    shutil.rmtree(p, ignore_errors=True) if os.path.isdir(p) else os.remove(p)
print("Final output:", sorted(os.listdir(WORK)))
